In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    matthews_corrcoef,
    roc_curve
)

# --------------------------------------------------
# LOAD FINAL EVALUATION RESULTS
# --------------------------------------------------

EVAL_SAVE = "/content/drive/MyDrive/VoxIntegrity_Data/dev_eval_A_checkpoint.npz"

data = np.load(EVAL_SAVE)

labels = data["labels"]
predictions = data["predictions"]
scores = data["scores"]

print("=" * 70)
print("FINE-TUNE A — FINAL DEV EVALUATION")
print("=" * 70)

print("Total samples:", len(labels))
print("Bonafide:", np.sum(labels == 0))
print("Spoof:", np.sum(labels == 1))


# --------------------------------------------------
# CONFUSION MATRIX
# --------------------------------------------------

cm = confusion_matrix(
    labels,
    predictions,
    labels=[0, 1]
)

TN, FP, FN, TP = cm.ravel()

print("\nConfusion Matrix:")
print(cm)

print("\nTN =", TN)
print("FP =", FP)
print("FN =", FN)
print("TP =", TP)


# --------------------------------------------------
# BASIC METRICS
# --------------------------------------------------

accuracy = accuracy_score(labels, predictions)

balanced_acc = balanced_accuracy_score(
    labels,
    predictions
)

precision = precision_score(
    labels,
    predictions,
    zero_division=0
)

recall = recall_score(
    labels,
    predictions,
    zero_division=0
)

f1 = f1_score(
    labels,
    predictions,
    zero_division=0
)

mcc = matthews_corrcoef(
    labels,
    predictions
)

roc_auc = roc_auc_score(
    labels,
    scores
)


# --------------------------------------------------
# ERROR RATES
# --------------------------------------------------

# Positive class = SPOOF

FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
FNR = FN / (FN + TP) if (FN + TP) > 0 else 0

# In anti-spoofing terminology:
# FAR = spoof incorrectly accepted as bonafide
# FRR = bonafide incorrectly rejected as spoof

FAR = FN / (FN + TP) if (FN + TP) > 0 else 0
FRR = FP / (FP + TN) if (FP + TN) > 0 else 0


# --------------------------------------------------
# EER
# --------------------------------------------------

fpr, tpr, thresholds = roc_curve(
    labels,
    scores,
    pos_label=1
)

fnr = 1 - tpr

eer_index = np.nanargmin(
    np.abs(fpr - fnr)
)

EER = (fpr[eer_index] + fnr[eer_index]) / 2


# --------------------------------------------------
# PRINT RESULTS
# --------------------------------------------------

print("\n" + "=" * 70)
print("FINAL METRICS")
print("=" * 70)

print(f"Accuracy          : {accuracy * 100:.2f}%")
print(f"Balanced Accuracy : {balanced_acc * 100:.2f}%")
print(f"Precision         : {precision * 100:.2f}%")
print(f"Recall            : {recall * 100:.2f}%")
print(f"F1 Score          : {f1 * 100:.2f}%")
print(f"MCC               : {mcc:.4f}")
print(f"ROC-AUC           : {roc_auc * 100:.2f}%")

print("\n--- Anti-Spoofing Error Rates ---")

print(f"FAR (Spoof → Bonafide) : {FAR * 100:.2f}%")
print(f"FRR (Bonafide → Spoof) : {FRR * 100:.2f}%")

print(f"\nEER               : {EER * 100:.2f}%")

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        labels,
        predictions,
        target_names=["Bonafide", "Spoof"],
        digits=4
    )
)

FINE-TUNE A — FINAL DEV EVALUATION
Total samples: 24844
Bonafide: 2548
Spoof: 22296

Confusion Matrix:
[[ 2349   199]
 [    3 22293]]

TN = 2349
FP = 199
FN = 3
TP = 22293

FINAL METRICS
Accuracy          : 99.19%
Balanced Accuracy : 96.09%
Precision         : 99.12%
Recall            : 99.99%
F1 Score          : 99.55%
MCC               : 0.9552
ROC-AUC           : 99.94%

--- Anti-Spoofing Error Rates ---
FAR (Spoof → Bonafide) : 0.01%
FRR (Bonafide → Spoof) : 7.81%

EER               : 0.91%

CLASSIFICATION REPORT
              precision    recall  f1-score   support

    Bonafide     0.9987    0.9219    0.9588      2548
       Spoof     0.9912    0.9999    0.9955     22296

    accuracy                         0.9919     24844
   macro avg     0.9949    0.9609    0.9771     24844
weighted avg     0.9919    0.9919    0.9917     24844

